# Flood Risk Prediction using XGBoost
**Author:** Ale Navarro

This notebook trains an XGBoost classifier using the shared `common.py` module so it is directly comparable with the other models in the project.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import joblib
import matplotlib.pyplot as plt
import common

from xgboost import XGBClassifier
from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay

## 1. Load data

In [ ]:
df = common.load_data()

X, y = common.build_features(df)

X_train, X_test, y_train, y_test = common.chronological_split(X, y)

print(f"Training samples: {len(X_train)}")
print(f"Testing samples: {len(X_test)}")

## 2. Train XGBoost

In [ ]:
model = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42
)

model.fit(X_train, y_train)

## 3. Predictions

In [ ]:
preds = model.predict(X_test)
probs = model.predict_proba(X_test)[:,1]

## 4. Evaluate

In [ ]:
baseline = common.persistence_baseline(df, y_test)

results = [
    baseline,
    common.evaluate("XGBoost", y_test, preds, probs)
]

common.comparison_table(results)

## 5. Confusion Matrix

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_test, preds)
plt.show()

## 6. ROC Curve

In [ ]:
RocCurveDisplay.from_predictions(y_test, probs)
plt.show()

## 7. Feature Importance

In [ ]:
importance = sorted(
    zip(X.columns, model.feature_importances_),
    key=lambda x: x[1],
    reverse=True
)

for feature, score in importance:
    print(f"{feature}: {score:.4f}")

## 8. Save model

In [ ]:
joblib.dump(model, "../backend/models/xgboost.joblib")
print("Model saved to ../backend/models/xgboost.joblib")

## Conclusions

- Uses the shared preprocessing from `common.py`.
- Uses the agreed chronological split.
- Evaluated with the common metrics for fair comparison.
- Saved as `xgboost.joblib` for dashboard integration.


In [ ]:
# Optuna hyperparameter optimisation for XGBoost
import optuna
from sklearn.metrics import f1_score
from xgboost import XGBClassifier

tscv = common.cv_splitter()

def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators",100,600),
        "max_depth": trial.suggest_int("max_depth",3,10),
        "learning_rate": trial.suggest_float("learning_rate",0.01,0.3,log=True),
        "subsample": trial.suggest_float("subsample",0.6,1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree",0.6,1.0),
        "min_child_weight": trial.suggest_int("min_child_weight",1,10),
        "random_state":42,
        "eval_metric":"logloss",
        "n_jobs":-1
    }
    scores=[]
    for tr,val in tscv.split(X_train):
        model=XGBClassifier(**params)
        model.fit(X_train.iloc[tr],y_train.iloc[tr])
        pred=model.predict(X_train.iloc[val])
        scores.append(f1_score(y_train.iloc[val],pred))
    return sum(scores)/len(scores)

study=optuna.create_study(direction="maximize")
study.optimize(objective,n_trials=50)
print(study.best_params)
print(study.best_value)


In [ ]:
best_params=study.best_params
model_xgb=XGBClassifier(**best_params,random_state=42,eval_metric='logloss',n_jobs=-1)
model_xgb.fit(X_train,y_train)

pred_xgb=model_xgb.predict(X_test)
prob_xgb=model_xgb.predict_proba(X_test)[:,1]

results=[
    common.evaluate("XGBoost (Optuna)",y_test,pred_xgb,prob_xgb),
    common.persistence_baseline(df,y_test)
]
common.comparison_table(results)
